# Tech Challenge IADT Fase 4 - Sistema Multimodal Completo

# Apresentacao: 15 minutos

## Objetivo

Este projeto implementa um **sistema de análise multimodal** para monitoramento médico, conforme o Tech Challenge da Fase 4.
Ele integra três fontes de dados e as combina em uma única decisão clínica:

- **Vídeo** (sessões de fisioterapia/cirurgia): postura com **OpenPose‑Light (MediaPipe)** + objetos/áreas críticas com **YOLOv8**.
- **Áudio** (consultas médicas): transcrição e análise de sentimento/entidades com **Azure Cognitive Services** (Speech to Text + Text Analytics), com detecção de alterações vocais (fadiga, falta de ar, dor).
- **Anomalias**: séries temporais de sinais vitais, evolução de prescrições e padrões de movimentação.

Os três módulos alimentam um **workflow integrado** que calcula uma **pontuação de risco** e gera o **alerta final à equipe médica**. O notebook roda em dois modos (escolhidos na execução): **offline** (dados simulados, sem chaves) ou **azure** (serviços reais).

In [ ]:
# =====================================================================
# SEÇÃO 0 — PREPARAÇÃO DO AMBIENTE E MODO DE EXECUÇÃO
# Nesta abertura nós carregamos as bibliotecas de visão computacional e
# dados que serão usadas ao longo da apresentação e definimos em qual
# modo o sistema vai rodar (offline para demonstração ou Azure real).
# =====================================================================
import cv2          # leitura e manipulação de vídeo/frames (Aula 01)
import numpy as np  # operações numéricas sobre imagens e séries temporais
import torch        # carrega o modelo YOLOv8 (Aulas de visão computacional)
import mediapipe as mp  # OpenPose-Light: análise postural (Aulas 02/03)
import pandas as pd # tabelas de sinais vitais (séries temporais)
from scipy import stats # apoio estatístico para detecção de anomalias
import os           # variáveis de ambiente e caminhos de arquivo

# ---- Configuracao de execucao (switch em tempo de execucao) ----
# Este é o ponto onde, na demonstração, informamos a origem dos dados:
# MODO = "offline"  -> sem Azure: usa transcricao/texto e frames simulados
# MODO = "azure"    -> usa Azure Cognitive Services (exige chaves e pacotes)
try:
    MODO = input("Modo de execucao [offline/azure] (Enter = offline): ").strip().lower() or "offline"
except Exception:
    MODO = os.getenv("MODO", "offline")
CAMINHO_VIDEO = os.getenv("CAMINHO_VIDEO", "dados/video.mp4")  # vídeo clínico real (opcional)
CAMINHO_AUDIO = os.getenv("CAMINHO_AUDIO", "dados/audio.wav")  # áudio de consulta (opcional)
AZURE_ON = (MODO == "azure")
print("MODO de execucao:", MODO)

# ---- MediaPipe Pose (OpenPose-Light) ----
# Inicializamos o detector de postura uma única vez; se a biblioteca não
# estiver disponível, seguimos em modo offline sem travar a apresentação.
try:
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,     # modo de vídeo (rastreio entre frames)
        model_complexity=1,          # equilíbrio entre precisão e velocidade
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    _mp_ok = True
except Exception:
    pose = None
    _mp_ok = False

In [ ]:
# ---------------------------------------------------------------------
# Modelo YOLOv8 — detecção de objetos e áreas críticas no vídeo clínico.
# Tentamos carregar o modelo pré-treinado; se não houver internet/pacote,
# o sistema segue em modo de demonstração (objetos simulados).
# ---------------------------------------------------------------------
try:
    model_yolo = torch.hub.load('ultralytics/yolov8', 'yolov8n', pretrained=True)
except Exception:
    model_yolo = None
_yolo_online = model_yolo is not None  # flag que controla real x simulado
print("YOLO disponivel:", _yolo_online)

In [ ]:
def detectar_objetos_yolo(frame):
    """
    Passa um frame pelo YOLOv8 e devolve a lista de objetos detectados
    (rótulo, confiança e caixa delimitadora). Na apresentação, serve para
    mostrar a detecção de instrumentos críticos e áreas de risco.
    """
    if _yolo_online and model_yolo is not None:
        results = model_yolo(frame)          # inferência do modelo no frame
        objetos = []
        for detection in results.xyxy[0]:    # varre cada detecção (bounding box)
            x1, y1, x2, y2, conf, cls = detection
            label = model_yolo.names[int(cls)]  # nome da classe prevista
            obj = {'label': label, 'confidence': float(conf)}
            obj['bbox'] = [int(x1), int(y1), int(x2), int(y2)]
            objetos.append(obj)
        return objetos
    else:
        # Modo offline: objetos simulados para demostracao
        # (garante que a dinâmica do pipeline funcione sem o modelo real)
        obj1 = {'label': 'instrumento_critico', 'confidence': 0.85, 'bbox': [100, 100, 200, 300]}
        obj2 = {'label': 'area_perigosa', 'confidence': 0.72, 'bbox': [300, 50, 400, 200]}
        return [obj1, obj2]

In [ ]:
def detectar_postura(frame):
    """
    Analisa a postura do paciente/profissional usando MediaPipe Pose
    (alternativa leve ao OpenPose). Retorna landmarks 3D de pontos-chave
    (joelhos, quadril, ombros etc.) usados depois para identificar
    movimentos fora do padrão clínico. Retorna SEMPRE um dict.
    """
    if _mp_ok and pose is not None:
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # MediaPipe usa RGB
        results = pose.process(image_rgb)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            keypoints = {}
            nomes = ['nariz', 'ldeltoide', 'rdeltoide', 'lquadril', 'rquadril',
                     'ljoelho', 'rjoelho', 'ltobillo', 'rtobillo']
            for name, lm in zip(nomes, landmarks):        # guarda x,y,z de cada ponto
                keypoints[name] = (lm.x, lm.y, lm.z)
            return {'status': 'sucesso', 'keypoints': keypoints}
    return {'status': 'nenhum_corpo_detectado'}

In [ ]:
def processar_video_clinico(caminho_video, max_frames=50):
    """
    Percorre o vídeo frame a frame combinando as duas análises de vídeo:
    YOLOv8 (objetos críticos) + MediaPipe (postura). Para cada frame gera
    um registro com score de anomalia e alertas. É o coração da
    'Análise de Vídeo' do Tech Challenge.
    """
    cap = cv2.VideoCapture(caminho_video)  # abre o arquivo de vídeo
    resultados = []
    frame_count = 0
    while cap.isOpened() and frame_count < max_frames:  # limita frames na demo
        ret, frame = cap.read()
        if not ret:
            break
        objetos = detectar_objetos_yolo(frame)  # objetos/áreas críticas
        postura = detectar_postura(frame)       # landmarks posturais
        score_anomalia = 0
        alertas = []
        if postura['status'] == 'sucesso':
            kpts = postura['keypoints']
            lje = kpts.get('ljoelho', (0,0,0))
            rje = kpts.get('rjoelho', (0,0,0))
            # Regras simples de postura inadequada (joelho muito esticado/dobrado)
            if lje[2] > 0.8 or rje[2] > 0.8:
                score_anomalia += 1
                alertas.append('Joelho esticada')
            if lje[2] < 0.2 or rje[2] < 0.2:
                score_anomalia += 1
                alertas.append('Joelho dobrado excessivamente')
        if len(objetos) > 0:
            # Considera críticos apenas os objetos com confiança > 0.5
            labels_criticos = [o['label'] for o in objetos if o['confidence'] > 0.5]
            alertas.append('Objetos detectados: %s' % ', '.join(labels_criticos[:3]))
            score_anomalia += len(labels_criticos)
        resultados.append({'frame': frame_count, 'objetos': len(objetos), 'postura_ok': postura['status'] == 'sucesso', 'score_anomalia': score_anomalia, 'alertas': '; '.join(alertas) if alertas else 'Nenhum'})
        frame_count += 1
    cap.release()
    return resultados

## 2. Analise de Audio Medico (5 min)

In [ ]:
# =====================================================================
# SEÇÃO 2 — ANÁLISE DE ÁUDIO (AZURE COGNITIVE SERVICES)
# Configuramos Speech to Text (transcrição) e Text Analytics
# (sentimento/entidades). Tudo é protegido: se as libs/chaves não
# existirem, o sistema cai em modo offline e a demonstração continua.
# =====================================================================
try:
    import azure.cognitiveservices.speech as speechsdk      # transcrição de fala
    from azure.ai.textanalytics import TextAnalyticsClient  # sentimento/entidades
    from azure.core.credentials import AzureKeyCredential
    _HAS_AZURE_LIBS = True
except Exception:
    _HAS_AZURE_LIBS = False

# As credenciais vêm de variáveis de ambiente (nunca hardcoded).
speech_key = os.getenv("AZURE_SPEECH_KEY", "")
speech_region = os.getenv("AZURE_SPEECH_REGION", "westus2")
text_key = os.getenv("AZURE_TEXT_ANALYTICS_KEY", "")
text_region = os.getenv("AZURE_TEXT_ANALYTICS_REGION", "westus2")

# _azure_on = true apenas se MODO=azure E libs E chaves presentes
_azure_on = bool(AZURE_ON and _HAS_AZURE_LIBS and speech_key and text_key)
if _azure_on:
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)
    speech_config.speech_recognition_language = "pt-BR"
    text_analytics_client = TextAnalyticsClient(
        endpoint=f"https://{text_region}.api.cognitive.microsoft.com/",
        credential=AzureKeyCredential(text_key),
    )
else:
    speech_config = None
    text_analytics_client = None

if AZURE_ON and not _azure_on:
    print("MODO=azure pedido, mas sem pacotes/chaves -> caindo em OFFLINE")
print("Azure configurado" if _azure_on else "Modo OFFLINE (sem Azure)")

In [ ]:
def transcrever_audio(caminho_audio):
    """
    Converte o áudio da consulta médica em texto (fala em texto, Aula 04)
    usando o Azure Speech to Text. Offline/sem arquivo -> string vazia, e
    o passo de execução usa uma transcrição simulada de exemplo.
    """
    if not _azure_on:
        return ""
    try:
        audio_config = speechsdk.AudioConfig(filename=caminho_audio)
        speech_recognizer = speechsdk.SpeechRecognizer(
            speech_config=speech_config, audio_config=audio_config)
        result = speech_recognizer.recognize_once()
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            return result.text  # texto transcrito da fala
        return ""
    except Exception as e:
        print(f"Erro na transcricao: {e}")
        return ""

In [ ]:
def analisar_texto_azure(texto):
    """
    Analisa o texto transcrito: sentimento (positivo/neutro/negativo) e
    extração de entidades relevantes. No Azure usa Text Analytics; offline
    aplica uma heurística simples por palavras-chave clínicas.
    """
    padrao = {'sentimento': {'rotulo': 'neutro',
                             'pontuacoes': {'positivo': 0.0, 'negativo': 0.0, 'neutro': 1.0}},
              'entidades': []}
    if not texto or len(texto.strip()) < 3:
        return padrao
    if not _azure_on:
        # Heurística offline: palavras ligadas a desconforto => "negativo"
        t = texto.lower()
        negativos = ['dor', 'cansado', 'fadiga', 'falta de ar', 'exausto',
                     'complicacao', 'desconforto', 'socorro']
        neg = any(w in t for w in negativos)
        return {'sentimento': {'rotulo': ('negativo' if neg else 'neutro'),
                               'pontuacoes': {'positivo': 0.0,
                                              'negativo': (1.0 if neg else 0.0),
                                              'neutro': (0.0 if neg else 1.0)}},
                'entidades': []}
    try:
        sentiment_result = text_analytics_client.analyze_sentiment([texto])[0]
        entity_result = text_analytics_client.recognize_entities([texto])[0]
    except Exception as e:
        print(f"Azure Text Analytics nao disponivel: {e}")
        return padrao
    # Filtra apenas categorias de interesse clínico
    entidades_medicas = []
    for entity in entity_result.entities:
        if entity.category in ['Person', 'Organization', 'Quantity', 'Temperature', 'DateTime']:
            entidades_medicas.append({'texto': entity.text, 'tipo': entity.category,
                                      'confianca': entity.confidence_score})
    cs = sentiment_result.confidence_scores
    return {'sentimento': {'rotulo': sentiment_result.sentiment,
                           'pontuacoes': {'positivo': float(cs.positive),
                                          'negativo': float(cs.negative),
                                          'neutro': float(cs.neutral)}},
            'entidades': entidades_medicas}

In [ ]:
def detectar_alteracoes_vocais(texto_transcrito):
    """
    Procura no texto da fala sinais indicativos de condições médicas
    (fadiga, dificuldade respiratória, dor), conforme o Tech Challenge.
    Retorna, por condição, se foi encontrada e com quais termos.
    """
    padroes_clinicos = {
        'fadiga': ['cansado', 'fadiga', 'sono', 'exausto'],
        'dificuldade_respiratoria': ['dificuldade para respirar', 'falta de ar'],
        'dor': ['dor', 'complicacao'],
    }
    texto_lower = texto_transcrito.lower() if texto_transcrito else ''
    resultados = {}
    for condicao, palavras in padroes_clinicos.items():
        encontradas = [p for p in palavras if p in texto_lower]  # casa termos-chave
        resultados[condicao] = {'presente': len(encontradas) > 0, 'encontradas': encontradas, 'quantidade': len(encontradas)}
    return resultados

In [ ]:
def detectar_anomalias_sinais_vitais(df_sinais):
    """
    Detecção de anomalias em séries temporais de sinais vitais.
    Usa o método "3 sigma" (média ± 3 desvios) para frequência cardíaca
    e um limite clínico fixo (SpO2 < 95%) para oxigenação — exatamente os
    exemplos citados no enunciado (batimentos e oxigenação).
    """
    anomalia = {'fc_anomalia': False, 'bp_anomalia': False, 'spo2_anomalia': False, 'alertas': []}
    if 'fc' in df_sinais.columns:
        fc = df_sinais['fc']
        fc_media = fc.mean()
        fc_desvio = fc.std()
        # Faixa de normalidade estatística (3 sigma)
        fc_limite_superior = fc_media + (3 * fc_desvio)
        fc_limite_inferior = fc_media - (3 * fc_desvio)
        fc_atual = df_sinais['fc'].iloc[-1]  # leitura mais recente
        if fc_atual > fc_limite_superior:
            anomalia['fc_anomalia'] = True
            anomalia['alertas'].append('FC CRITICA: %.1f bpm' % fc_atual)
        elif fc_atual < fc_limite_inferior:
            anomalia['fc_anomalia'] = True
            anomalia['alertas'].append('FC CRITICA: %.1f bpm' % fc_atual)
        else:
            anomalia['alertas'].append('FC normal: %.1f bpm' % fc_atual)
    if 'spo2' in df_sinais.columns:
        spo2 = df_sinais['spo2']
        spo2_atual = spo2.iloc[-1]
        # Hipoxemia: oxigenação abaixo de 95% é sinal de risco
        if spo2_atual < 95:
            anomalia['spo2_anomalia'] = True
            anomalia['alertas'].append('SpO2 BAIXA: %.1f%%' % spo2_atual)
        else:
            anomalia['alertas'].append('SpO2 normal: %.1f%%' % spo2_atual)
    return anomalia

In [ ]:
def detectar_anomalias_prescricoes(lista_prescricoes):
    """
    Compara a prescrição atual com a anterior para identificar alterações
    inesperadas no tratamento: novo medicamento, mudança de dosagem e um
    possível risco de interação quando ambos ocorrem juntos.
    """
    anomalia = {'medicamento_novo': False, 'dosagem_alterada': False, 'interacao_drogas': False, 'alertas': []}
    if len(lista_prescricoes) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    atual = lista_prescricoes[-1]
    anterior = lista_prescricoes[-2]
    # Conjunto de medicamentos (normalizado em minúsculas) de cada prescrição
    keys_atual = set(k.lower() for k in atual.keys())
    keys_anterior = set(k.lower() for k in anterior.keys())
    novos = keys_atual - keys_anterior  # medicamentos que acabaram de entrar
    if novos:
        anomalia['medicamento_novo'] = True
        anomalia['alertas'].append('Novo medicamento: %s' % ', '.join([k for k in atual.keys() if k.lower() in novos]))
    comuns = keys_atual & keys_anterior  # presentes nas duas
    for med in comuns:
        # Mesmo medicamento com dosagem diferente => alteração de tratamento
        if atual.get(med) != anterior.get(med):
            anomalia['dosagem_alterada'] = True
            anomalia['alertas'].append('Dosagem alterada para: %s' % med)
    # Novo medicamento + mudança de dosagem levantam suspeita de interação
    if anomalia['medicamento_novo'] and anomalia['dosagem_alterada']:
        anomalia['interacao_drogas'] = True
        anomalia['alertas'].append('WARNING: Possivel interacao: novo medicamento + cambio de dosagem')
    return anomalia

In [ ]:
def detectar_anomalias_movimentacao(keypoints_history):
    """
    Analisa padrões de movimentação do paciente a partir dos keypoints de
    postura ao longo do tempo. Mede o deslocamento do centroide do quadril
    frame a frame e sinaliza movimento repentino, imobilidade anormal e o
    padrão suspeito de queda (movimento brusco seguido de imobilidade).
    """
    anomalia = {'movimento_repetino': False, 'imobilidade_anormal': False, 'quedas_suspeitas': False, 'alertas': []}
    if len(keypoints_history) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    # Centroide (ponto médio) entre os dois quadris = proxy da posição do corpo
    centoidos = []
    for kp_data in keypoints_history:
        lq = kp_data.get('lquadril', (0, 0))
        rq = kp_data.get('rquadril', (0, 0))
        centroide = ((lq[0] + rq[0]) / 2, (lq[1] + rq[1]) / 2)
        centoidos.append(centroide)
    if len(centoidos) >= 2:
        # Distância percorrida entre frames consecutivos = velocidade relativa
        velocidades = []
        for i in range(1, len(centoidos)):
            dist = ((centoidos[i][0] - centoidos[i-1][0])**2 + (centoidos[i][1] - centoidos[i-1][1])**2)**0.5
            velocidades.append(dist)
        if velocidades:
            velocidade_max = max(velocidades)
            velocidade_media = sum(velocidades) / len(velocidades)
            # Pico muito acima da média indica movimento repentino
            if velocidade_max > 3 * velocidade_media and velocidade_media > 0:
                anomalia['movimento_repetino'] = True
                anomalia['alertas'].append('Movimento repentino: dist %.3f' % velocidade_max)
            # Pouco movimento nas últimas leituras => imobilidade anormal
            N = min(10, len(velocidades))
            ultimas = velocidades[-N:]
            vm = sum(ultimas) / len(ultimas) if ultimas else 0
            if vm < 0.05 and len(ultimas) >= 10:
                anomalia['imobilidade_anormal'] = True
                anomalia['alertas'].append('Imobilidade: %d frames sem movimento' % len([v for v in ultimas if v < 0.01]))
            # Combinação dos dois sugere possível queda
            if anomalia['movimento_repetino'] and anomalia['imobilidade_anormal']:
                anomalia['quedas_suspeitas'] = True
                anomalia['alertas'].append('WARNING: Padrao de queda suspeito: movimento repentino + imobilidade')
    return anomalia

In [ ]:
def workflow_integrado(resultado_video, resultado_audio, resultado_sinais):
    """
    INTEGRAÇÃO MULTIMODAL — consolida vídeo + áudio + anomalias em uma
    única decisão clínica. Cada fonte contribui com pontos de risco
    (vídeo=2 por alerta, áudio=3 por alteração, sinais=3), e a soma é
    convertida em um nível (Normal / ALTO / CRÍTICO) com ação recomendada.
    """
    alertas_gerais = []
    punteis_risco = 0
    # --- contribuição do VÍDEO: alertas de objetos/postura ---
    if resultado_video and 'total_alertas' in resultado_video:
        vs = resultado_video.get('alertas_listas', [])
        sig = [a for a in vs if 'Nenhum' not in a]
        if sig:
            alertas_gerais.extend(sig)
            punteis_risco += len(sig) * 2
    # --- contribuição do ÁUDIO: alterações vocais detectadas ---
    if resultado_audio and 'total_alteracoes' in resultado_audio:
        aps = [k for k, v in resultado_audio['alteracoes'].items() if v.get('presente', False)]
        if aps:
            # Traduz cada código de alteração em uma mensagem legível para a equipe
            mapa = {'fadiga': 'Paciente relata fadiga', 'dificuldade_respiratoria': 'Paciente relata dificuldade respiratoria', 'dor': 'Paciente relata dor', 'alerta_urgencia': 'ALERTA URGENCIA: solicitacao de ajuda'}
            for a in aps:
                m = mapa.get(a, a)
                alertas_gerais.append(m)
                punteis_risco += 3
    # --- contribuição dos SINAIS VITAIS/prescrições/movimentação ---
    if resultado_sinais and 'alertas' in resultado_sinais:
        alertas_gerais.extend(resultado_sinais['alertas'])
        punteis_risco += 3
    # --- decisão final pela pontuação de risco acumulada ---
    if punteis_risco >= 6:
        decisao = 'ALERTA MÉDICO IMEDIATO'
        nivel = 'CRITICO'
        acao = 'Contatar equipe médica imediatamente'
    elif punteis_risco >= 3:
        decisao = 'ALERTA - Revisao necessaria'
        nivel = 'ALTO'
        acao = 'Avaliar paciente e monitorar 30 min'
    else:
        decisao = 'Monitoramento normal continuo'
        nivel = 'Normal'
        acao = 'Continuar rotina de cuidados'
    return {'decisao': decisao, 'nivel': nivel, 'acao_recomendada': acao, 'total_alertas': len(alertas_gerais), 'alertas': '; '.join(alertas_gerais) if alertas_gerais else 'Nenhum motivo de alerta', 'punteis_risco': punteis_risco}

## 4. Execução do Pipeline Multimodal (vídeo + áudio + anomalias)
Gera `resultado_video`, `resultado_audio` e `resultado_sinais` que alimentam o relatório/decisão final.

In [ ]:
# =====================================================================
# EXECUÇÃO — ANÁLISE DE VÍDEO
# Aqui a demonstração dispara a análise de vídeo de verdade. Se houver um
# arquivo real (CAMINHO_VIDEO), processamos frame a frame; caso contrário
# (modo offline), usamos frames sintéticos para evidenciar YOLO+MediaPipe
# sem depender de mídia. O resultado é agregado no formato que o
# workflow_integrado espera (com 'total_alertas' e 'alertas_listas').
# =====================================================================
def gerar_frames_sinteticos(n=40):
    """Frames simulados para rodar YOLO/MediaPipe sem arquivo real."""
    frames = []
    for i in range(n):
        base = np.full((480, 640, 3), 60 + i, dtype=np.uint8)
        frames.append(base)
    return frames

def executar_video():
    resultados = None
    if os.path.exists(CAMINHO_VIDEO):  # vídeo clínico real disponível?
        try:
            resultados = processar_video_clinico(CAMINHO_VIDEO)
        except Exception as e:
            print("Falha no video real, usando frames sinteticos:", e)
            resultados = None
    if not resultados:
        # Fallback offline: aplica os MESMOS detectores sobre frames simulados
        resultados = []
        for idx, frame in enumerate(gerar_frames_sinteticos()):
            objetos = detectar_objetos_yolo(frame)
            postura = detectar_postura(frame)
            alertas, score = [], 0
            if postura.get('status') == 'sucesso':
                kpts = postura.get('keypoints', {})
                lje = kpts.get('ljoelho', (0, 0, 0)); rje = kpts.get('rjoelho', (0, 0, 0))
                if lje[2] > 0.8 or rje[2] > 0.8:
                    score += 1; alertas.append('Joelho esticado')
                if lje[2] < 0.2 or rje[2] < 0.2:
                    score += 1; alertas.append('Joelho dobrado excessivamente')
            if objetos:
                labels = [o['label'] for o in objetos if o['confidence'] > 0.5]
                if labels:
                    score += len(labels)
                    alertas.append('Objetos detectados: ' + ', '.join(labels[:3]))
            resultados.append({'frame': idx, 'objetos': len(objetos),
                               'postura_ok': postura.get('status') == 'sucesso',
                               'score_anomalia': score,
                               'alertas': '; '.join(alertas) if alertas else 'Nenhum'})
    # Agrega os resultados por frame nos campos usados pela integração
    alertas_listas = [r['alertas'] for r in resultados if r['alertas'] != 'Nenhum']
    return {'frames': len(resultados),
            'objetos_totais': sum(r['objetos'] for r in resultados),
            'anomalias_posturais': sum(r['score_anomalia'] for r in resultados),
            'alertas_listas': alertas_listas,
            'total_alertas': len(alertas_listas)}

resultado_video = executar_video()
print("Video: {} frames | {} alertas".format(resultado_video['frames'],
                                              resultado_video['total_alertas']))

In [ ]:
# =====================================================================
# EXECUÇÃO — ANÁLISE DE ÁUDIO
# Encadeia as etapas de fala: transcreve o áudio (ou usa uma transcrição
# simulada no modo offline), analisa sentimento/entidades e detecta
# alterações vocais. O pacote agregado atende ao contrato do
# workflow_integrado ('alteracoes' + 'total_alteracoes').
# =====================================================================
def executar_audio():
    texto = ""
    if os.path.exists(CAMINHO_AUDIO):          # tenta transcrever áudio real
        texto = transcrever_audio(CAMINHO_AUDIO)
    if not texto:                              # offline/sem arquivo: exemplo
        texto = ("Paciente: estou muito cansado e com falta de ar. "
                 "Sinto dor no peito desde ontem.")
    analise = analisar_texto_azure(texto)      # sentimento + entidades
    alteracoes = detectar_alteracoes_vocais(texto)  # fadiga/falta de ar/dor
    return {'transcricao': texto,
            'sentimento': analise.get('sentimento', {}),
            'entidades': analise.get('entidades', []),
            'alteracoes': alteracoes,
            'total_alteracoes': sum(1 for v in alteracoes.values() if v.get('presente'))}

resultado_audio = executar_audio()
print("Audio: sentimento={} | {} alteracoes vocais".format(
    resultado_audio['sentimento'].get('rotulo', 'neutro'),
    resultado_audio['total_alteracoes']))

In [ ]:
# =====================================================================
# EXECUÇÃO — DETECÇÃO DE ANOMALIAS
# Alimenta os três detectores de anomalia com dados de exemplo (séries de
# sinais vitais, evolução de prescrições e histórico de keypoints) e
# consolida todos os alertas em 'resultado_sinais', pronto para a
# decisão final do workflow integrado.
# =====================================================================
def executar_sinais():
    # Série temporal de batimentos (fc) e oxigenação (spo2) com um pico anômalo
    df_sinais = pd.DataFrame({
        'fc':   [72, 75, 74, 78, 80, 82, 130],
        'spo2': [98, 98, 97, 96, 96, 95, 92],
    })
    resultado_sinais = detectar_anomalias_sinais_vitais(df_sinais)
    # Comparação entre duas prescrições (surge novo medicamento e muda dosagem)
    prescricoes = detectar_anomalias_prescricoes([
        {'Dipirona': '500mg'},
        {'Dipirona': '1g', 'Oxigenio': '2L/min'},
    ])
    # Histórico de quadris: salto brusco de posição (possível movimento repentino)
    movimentacao = detectar_anomalias_movimentacao([
        {'lquadril': (0.5, 0.5), 'rquadril': (0.6, 0.5)},
        {'lquadril': (0.9, 0.9), 'rquadril': (1.0, 0.9)},
        {'lquadril': (0.9, 0.9), 'rquadril': (1.0, 0.9)},
    ])
    # Junta os alertas das três fontes de anomalia em uma única lista
    alertas = list(resultado_sinais.get('alertas', []))
    alertas += prescricoes.get('alertas', []) + movimentacao.get('alertas', [])
    resultado_sinais['prescricoes'] = prescricoes
    resultado_sinais['movimentacao'] = movimentacao
    resultado_sinais['alertas'] = alertas
    return resultado_sinais

resultado_sinais = executar_sinais()
print("Sinais/anomalias ->", resultado_sinais['alertas'])

# Relatório Final para Apresentacao de 15 Minutos

In [ ]:
# =====================================================================
# RELATÓRIO FINAL (para a apresentação de 15 minutos)
# Resume os resultados das três análises e produz a decisão clínica final
# chamando o workflow_integrado com tudo que foi processado.
# =====================================================================
print('=' * 70)
print('TECH CHALLENGE IADT FASE 4 - SISTEMA MULTIMODAL COMPLETO')
print('Modo de execucao:', MODO.upper())
print('=' * 70)

# --- Resumo da ANÁLISE DE VÍDEO ---
print('\n## 1. DEMONSTRAÇÃO DE ANÁLISE DE VÍDEO')
print('   • OpenPose-Light/MediaPipe: analise postural em {} frames'.format(resultado_video['frames']))
print('   • YOLOv8: {} objetos/areas criticos detectados'.format(resultado_video['objetos_totais']))
print('   • Anomalias posturais (score): {}'.format(resultado_video['anomalias_posturais']))
print('   • Alertas de video: {}'.format(resultado_video['total_alertas']))

# --- Resumo da ANÁLISE DE ÁUDIO ---
print('\n## 2. DEMONSTRAÇÃO DE ANÁLISE DE ÁUDIO')
print('   • Transcricao: "{}"'.format(resultado_audio['transcricao'][:80]))
print('   • Sentimento: {}'.format(resultado_audio['sentimento'].get('rotulo', 'neutro')))
print('   • Entidades medicas: {}'.format(len(resultado_audio['entidades'])))
print('   • Alteracoes vocais: {}'.format(resultado_audio['total_alteracoes']))

# --- Resumo da DETECÇÃO DE ANOMALIAS ---
print('\n## 3. DETECÇÃO DE ANOMALIAS')
print('   • Sinais/prescricoes/movimentacao ->', '; '.join(resultado_sinais.get('alertas', ['nenhuma'])))

# --- INTEGRAÇÃO: gera o alerta final ---
print('\n## 4. INTEGRAÇÃO E DECISÃO FINAL')
resultado = workflow_integrado(resultado_video, resultado_audio, resultado_sinais)
print('   • Decisao:', resultado['decisao'])
print('   • Nivel:', resultado['nivel'])
print('   • Acao:', resultado['acao_recomendada'])
print('   • Alertas gerados:', resultado['total_alertas'])
print('   • Pontuacao de risco:', resultado['punteis_risco'])
print('   • Detalhe:', resultado['alertas'])

print('\n' + '=' * 70)
print('FIM DA APRESENTAÇÃO DO TECH CHALLENGE IADT FASE 4')
print('=' * 70)